In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121 -q
!pip install matplotlib opencv-python tqdm lxml -q
!pip install ultralytics -q
!pip install scikit-learn==1.7.2 -q
!pip show scikit-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 81.5 MB/s eta 0:00:00
Name: scikit-learn
Version: 1.7.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: cuml-cu12, fastai, hdbscan, imbalanced-learn, libpysal, librosa, mlxtend, pynndescent, sentence-transformers, shap, sklearn-pandas, tsfresh, umap-learn, yellowbrick


In [ ]:
import os
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---
DATA_ROOT = '/content/drive/MyDrive/Yolo_everything/Data_split'   # 👈 your main folder
OUTPUT_DIR = os.path.join(DATA_ROOT, "checkpoints")

NUM_CLASSES = 6  # 5 WBC types + background
NUM_EPOCHS = 20
BATCH_SIZE = 2
LEARNING_RATE = 0.005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- DATASET CLASS ---
class WBCDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        self.imgs = sorted(os.listdir(self.img_dir))
        self.annots = sorted(os.listdir(self.ann_dir))

        self.class_to_idx = {
            "Marcophage/Monocyte": 0,
            "Neutrophil": 1,
            "Eosinophil": 2,
            "Lymphocyte": 3,
            "Unknown cell/Debris": 4
        }

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        ann_path = os.path.join(self.ann_dir, self.annots[idx])

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            cls = obj.find("name").text.strip()
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx.get(cls, 0))

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)


# --- DATA LOADERS ---
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = WBCDataset(
    os.path.join(DATA_ROOT, "images/train"),
    os.path.join(DATA_ROOT, "annotations/train"),
    transforms=transform
)

val_dataset = WBCDataset(
    os.path.join(DATA_ROOT, "images/val"),
    os.path.join(DATA_ROOT, "annotations/val"),
    transforms=transform
)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)


# --- MODEL SETUP ---
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="COCO_V1")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model(NUM_CLASSES).to(DEVICE)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, weight_decay=0.0005)


# --- TRAINING + VALIDATION LOOPS ---
def train_one_epoch(model, optimizer, loader, device, epoch):
    model.train()
    running_loss = 0.0
    for images, targets in tqdm(loader, desc=f"Epoch {epoch+1} [Train]", ncols=100):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    avg_loss = running_loss / len(loader)
    print(f"🟢 Train Loss (Epoch {epoch+1}): {avg_loss:.4f}")
    return avg_loss


@torch.no_grad()
def validate(model, loader, device, epoch):
    model.eval()
    # In evaluation mode with targets, the model returns a list of detections per image.
    # Calculating a simple loss in the same way as training is not appropriate here.
    # This function is typically used for calculating evaluation metrics (e.g., mAP).
    print(f"🔵 Running validation (Epoch {epoch+1}) for inference and potential metric calculation.")
    for images, targets in tqdm(loader, desc=f"Epoch {epoch+1} [Val]", ncols=100):
        images = [img.to(device) for img in images]
        # Note: targets are not used by the model in eval mode for loss calculation.
        # They would be used with an evaluation metric function (e.g., COCOeval).
        outputs = model(images)
        # Process outputs (e.g., visualize predictions, calculate metrics)

    # Returning a placeholder value as no loss is calculated in this modified function
    return 0.0


# --- MAIN TRAINING LOOP ---
best_val_loss = float("inf") # Keep track of a validation metric if implemented

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch)

    # Removed the val_loss calculation as it's not done in the modified validate function
    validate(model, val_loader, DEVICE, epoch)

    # Save the model after each epoch (or based on an evaluation metric if implemented)
    # The checkpoint name now reflects only the epoch.
    checkpoint_path = os.path.join(OUTPUT_DIR, f"fasterrcnn_epoch{epoch+1}.pth")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"✅ Model saved: {checkpoint_path}")

# If you implement evaluation metrics in validate, you would use best_val_loss here.
print("\n🎯 Training complete!")

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# === SETTINGS ===
MODEL_PATH = '/content/drive/MyDrive/Yolo_everything/Data_split/checkpoints/fasterrcnn_epoch11.pth'
NUM_CLASSES = 6  # 5 WBC types + background
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === RECREATE MODEL ===
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# === LOAD TRAINED WEIGHTS ===
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print("✅ Model loaded successfully!")


In [ ]:
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T

# === Load image ===
IMG_PATH ='/content/drive/MyDrive/Yolo_everything/Data_split/images/test/P01_12_14_184.png'

# === Prepare image ===
image = Image.open(IMG_PATH).convert("RGB")
transform = T.Compose([T.ToTensor()])
img_tensor = transform(image).to(DEVICE)

# === Inference ===
with torch.no_grad():
    prediction = model([img_tensor])[0]

print("Predicted boxes:", prediction["boxes"])
print("Predicted labels:", prediction["labels"])
print("Scores:", prediction["scores"])


In [ ]:
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

# --- CONFIGURATION ---
DATA_ROOT = '/content/drive/MyDrive/Yolo_everything/Data_split'  # your dataset folder
MODEL_PATH = '/content/drive/MyDrive/Yolo_everything/Data_split/checkpoints/fasterrcnn_epoch11.pth'  # path to best model
OUTPUT_DIR = os.path.join(DATA_ROOT, "results_visualized")

os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES = 6  # 5 WBC types + background
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CLASS NAMES ---
class_names = ["background", "Marcophage/Monocyte", "Neutrophil", "Eosinophil", "Lymphocyte", "Unknown cell/Debris"]

# --- LOAD MODEL ---
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

print("🔹 Loading model...")
model = get_model(NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print("✅ Model loaded successfully!\n")

# --- IMAGE TRANSFORM ---
transform = transforms.Compose([transforms.ToTensor()])

# --- TEST IMAGE DIRECTORY ---
TEST_IMG_DIR = os.path.join(DATA_ROOT, "images/test")
image_files = [f for f in os.listdir(TEST_IMG_DIR) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

print(f"📸 Found {len(image_files)} test images")

# --- INFERENCE LOOP ---
for img_name in tqdm(image_files, desc="Running inference"):
    img_path = os.path.join(TEST_IMG_DIR, img_name)
    image = Image.open(img_path).convert("RGB")

    img_tensor = transform(image).to(DEVICE)

    # --- PREDICTION ---
    with torch.no_grad():
        prediction = model([img_tensor])[0]

    boxes = prediction["boxes"].detach().cpu().numpy()
    labels = prediction["labels"].detach().cpu().numpy()
    scores = prediction["scores"].detach().cpu().numpy()

    # --- VISUALIZATION ---
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)

    for box, label, score in zip(boxes, labels, scores):
        if score > 0.6:  # confidence threshold
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, ymin - 5, f"{class_names[label]} ({score:.2f})",
                    color='white', fontsize=10, weight='bold', backgroundcolor='black')

    plt.axis("off")
    output_path = os.path.join(OUTPUT_DIR, img_name)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

print(f"\n✅ Inference complete! Visualized images saved to:\n{OUTPUT_DIR}")


In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# === SETTINGS ===
MODEL_PATH = '/content/drive/MyDrive/Yolo_everything/Data_split/checkpoints/fasterrcnn_epoch11.pth'
NUM_CLASSES = 6  # 5 WBC types + background
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === RECREATE MODEL ===
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# === LOAD TRAINED WEIGHTS ===
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print("✅ Model loaded successfully!")


In [ ]:
!pip install torchmetrics scikit-learn matplotlib tqdm


In [ ]:
import os
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)
import torchvision.ops as ops
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---
DATA_ROOT = '/content/drive/MyDrive/Yolo_everything/Data_split'
MODEL_PATH = '/content/drive/MyDrive/Yolo_everything/Data_split/checkpoints/fasterrcnn_epoch11.pth'
OUTPUT_DIR = os.path.join(DATA_ROOT, "evaluation_results")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6  # 5 classes + background
CONF_THRESHOLD = 0.5
IOU_THRESHOLD = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

class_names = [
    "background",
    "Marcophage/Monocyte",
    "Neutrophil",
    "Eosinophil",
    "Lymphocyte",
    "Unknown cell/Debris",
]


# --- DATASET CLASS ---
class WBCDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms
        self.imgs = sorted(os.listdir(img_dir))
        self.annots = sorted(os.listdir(ann_dir))
        self.class_to_idx = {
            "Marcophage/Monocyte": 0,
            "Neutrophil": 1,
            "Eosinophil": 2,
            "Lymphocyte": 3,
            "Unknown cell/Debris": 4,
        }

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        ann_path = os.path.join(self.ann_dir, self.annots[idx])
        img = Image.open(img_path).convert("RGB")

        tree = ET.parse(ann_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            cls = obj.find("name").text.strip()
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx.get(cls, 0))

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.imgs)


# --- MODEL LOADING ---
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


print("🔹 Loading model...")
model = get_model(NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print("✅ Model loaded successfully!\n")

# --- DATA LOADING ---
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = WBCDataset(
    os.path.join(DATA_ROOT, "images/test"),
    os.path.join(DATA_ROOT, "annotations/test"),
    transforms=transform,
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
print(f"📁 Loaded {len(test_dataset)} test images.\n")

# --- TORCHMETRICS: mAP EVALUATION ---
metric = MeanAveragePrecision(iou_type="bbox")
y_true, y_pred = [], []

for images, targets in tqdm(test_loader, desc="Evaluating"):
    images = [img.to(DEVICE) for img in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

    with torch.no_grad():
        preds = model(images)

    metric.update(preds, targets)

    pred_boxes = preds[0]["boxes"].detach().cpu()
    pred_labels = preds[0]["labels"].detach().cpu()
    pred_scores = preds[0]["scores"].detach().cpu()

    gt_boxes = targets[0]["boxes"].detach().cpu()
    gt_labels = targets[0]["labels"].detach().cpu()

    # Filter predictions by confidence
    keep = pred_scores >= CONF_THRESHOLD
    pred_boxes = pred_boxes[keep]
    pred_labels = pred_labels[keep]

    # Match boxes using IoU
    if len(pred_boxes) > 0 and len(gt_boxes) > 0:
        ious = ops.box_iou(pred_boxes, gt_boxes)
        for i, pb in enumerate(pred_boxes):
            max_iou, match_idx = torch.max(ious[i], dim=0)
            if max_iou >= IOU_THRESHOLD:
                y_true.append(gt_labels[match_idx].item())
                y_pred.append(pred_labels[i].item())
            else:
                y_true.append(0)
                y_pred.append(pred_labels[i].item())

        unmatched_gt = set(range(len(gt_boxes))) - set(torch.argmax(ious, dim=0).tolist())
        for idx in unmatched_gt:
            y_true.append(gt_labels[idx].item())
            y_pred.append(0)
    elif len(gt_boxes) > 0:  # No predictions → all missed
        for lbl in gt_labels:
            y_true.append(lbl.item())
            y_pred.append(0)
    elif len(pred_boxes) > 0:  # No ground truth → all false positives
        for lbl in pred_labels:
            y_true.append(0)
            y_pred.append(lbl.item())


# --- Compute mAP Metrics ---
results = metric.compute()
map_50 = results["map_50"].item()
map_75 = results["map_75"].item()
map_5095 = results["map"].item()

print("\n📊 COCO-Style Evaluation Metrics:")
print(f"mAP@50:      {map_50:.4f}")
print(f"mAP@75:      {map_75:.4f}")
print(f"mAP@[50-95]: {map_5095:.4f}")

# --- Compute Per-Class Precision / Recall / F1 ---
y_true = np.array(y_true)
y_pred = np.array(y_pred)

per_class_precision = precision_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
per_class_recall = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📈 Per-Class Precision / Recall:")
metrics_table = []
for i, cname in enumerate(class_names[1:]):  # skip background
    print(f"{cname:22s} Precision: {per_class_precision[i]:.3f} | Recall: {per_class_recall[i]:.3f}")
    metrics_table.append({
        "Class": cname,
        "Precision": per_class_precision[i],
        "Recall": per_class_recall[i],
        "F1": 2 * (per_class_precision[i] * per_class_recall[i]) / (per_class_precision[i] + per_class_recall[i] + 1e-6)
    })

print(f"\nOverall Macro F1-score: {f1_macro:.4f}")

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3 ,4])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names[1:])
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix (Detections)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.pdf"), dpi=300)
plt.show()

# --- Export Metrics to CSV ---
metrics_df = pd.DataFrame(metrics_table)
metrics_df.loc[len(metrics_df.index)] = {
    "Class": "Overall",
    "Precision": np.mean(per_class_precision),
    "Recall": np.mean(per_class_recall),
    "F1": f1_macro
}
metrics_df["mAP@50"] = map_50
metrics_df["mAP@75"] = map_75
metrics_df["mAP@[50-95]"] = map_5095
metrics_df.to_csv(os.path.join(OUTPUT_DIR, "evaluation_metrics.csv"), index=False)

print(f"\n✅ Evaluation complete! Results saved in:\n{OUTPUT_DIR}")
print("📂 - evaluation_metrics.csv")
print("🖼️ - confusion_matrix.pdf")

In [ ]:
torch.save(model.state_dict(), "fasterrcnn_final.pth")
